# 3. A Rede de Monitoramento Brasileira


<div style="text-align: justify">Foram contabilizadas 479 estações de monitoramento da qualidade do ar no Brasil no ano de 2024, representando um acréscimo de 84 estações em relação ao levantamento realizado no ano de 2023 (BRASIL, 2023). Deste montante, 385 são estações que utilizam método de referência ou equivalente, um acréscimo de 27 unidades (aumento de 7,5%) em relação ao ano anterior. Já o monitoramento indicativo teve uma ampliação de 57 unidades, atingindo um total de 94 estações (aumento de 154%).</div><br/>
<div style="text-align: justify">O acréscimo significativo de unidades de monitoramento está associado ao maior número de respostas das UFs ao questionário aplicado pelo MMA, uma vez que identificaram mais estações de referência ou equivalentes que já operavam, mas não foram contabilizadas, e à implantação de estações de monitoramento indicativas. Considerando as respostas do questionário aplicado em 2023, apenas dois estados, Acre e Mato Grosso, afirmaram possuir um total de 37 estações de monitoramento indicativas instaladas. Enquanto isso, em 2024, mais 6 UFs (Amazonas, Amapá, Goiás, Maranhão, Pará e Tocantins) informaram a operação de estações indicativas, atingindo a marca de 94 equipamentos em operação.</div>

```{note}
É importante destacar que as estações indicativas são integradas em plataformas internacionais com finalidade científica, exploratória e informativa. A gestão dos dados ocorre através da integração dos equipamentos na plataforma, sem qualquer tratamento da informação. Em alguns casos, a operação e supervisão é realizada pelos OEMAs. Estas estações são equipadas com instrumentos e sensores não considerados equivalentes às estações de referência. As estações indicativas são capazes de monitorar a concentração de alguns poluentes atmosféricos em tempo real, no entanto, podem apresentar um grau de incerteza relevante em relação ao dado gerado, principalmente quando não são calibradas e operadas adequadamente. 
```

<div style="text-align: justify">Convém ressaltar, ainda, que as estações indicativas não atendem aos critérios estabelecidos pelo Guia Técnico para o Monitoramento e Avaliação da Qualidade do Ar, mas podem fornecer informações relevantes sobre a qualidade do ar, principalmente em locais sem nenhuma estação de referência ou equivalente.</div><br/>
<div style="text-align: justify">A Tabela 5 apresenta o número de estações de monitoramento da qualidade do ar reportadas pelos OEMAs nos anos de 2023 e 2024, bem como a variação no período. </div>
<br>
<br>

In [111]:
import os
import pandas as pd 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML
import numpy as np


# Caminho para a pasta de dados
rootPath = os.path.dirname(os.getcwd())

# Lendo o csv
aqmData = pd.read_csv(rootPath+'/data/Monitoramento_QAr_BR.csv')

df = aqmData.copy()

# Quebrar ANOS_MONITORADOS em listas
df["ANOS_MONITORADOS"] = df["ANOS_MONITORADOS"].str.split(",")

# Explodir para cada ano virar uma linha
df_exploded = df.explode("ANOS_MONITORADOS")

# Converter para inteiro
#df_exploded["ANOS_MONITORADOS"] = df_exploded["ANOS_MONITORADOS"].astype(int)

# Agrupar também por POLUENTE
df_grouped = (
    df_exploded.groupby(["POLUENTE", "UF", "ANOS_MONITORADOS"])
    .size()   # conta linhas
    .reset_index(name="NSTATION")
)

df_grouped['ANOS_MONITORADOS'] = df_grouped['ANOS_MONITORADOS'].astype(int)

# Lista de poluentes e UFs
poluentes = sorted(df_grouped["POLUENTE"].unique())
ufs = sorted(df_grouped["UF"].unique())


# Cores fixas por UF
color_map = {uf: f"hsl({i*40 % 360},70%,50%)" for i, uf in enumerate(ufs)}

fig = go.Figure()
trace_visibility = []

## Track UF que já teve legenda
#uf_legend_shown = {uf: False for uf in ufs}

# Criar todos os traços: um trace por UF e poluente
for pol in poluentes:
    df_pol = df_grouped[df_grouped["POLUENTE"] == pol]
    for uf in ufs:
        df_uf = df_pol[df_pol["UF"] == uf]
        y_values = df_uf["NSTATION"].values
        x_values = df_uf["ANOS_MONITORADOS"].values 

        fig.add_trace(
            go.Bar(
                x=x_values,
                y=y_values,
                name=uf,
                marker=dict(color=color_map[uf]),
                legendgroup=uf,
                showlegend=True,  # apenas um trace de cada UF aparece na legenda
                visible=(pol == poluentes[0]),  # primeiro poluente aparece inicialmente
                text=y_values,
                textposition="outside"
            )
        )
        #uf_legend_shown[uf] = True
        trace_visibility.append(pol)

# Dropdown menu
buttons = []
for pol in poluentes:
    vis = [p == pol for p in trace_visibility]
    buttons.append(dict(
        label=pol,
        method="update",
        args=[{"visible": vis},
              {"title": f"Monitoramento de {pol}"}]
    ))
anos = np.arange(df_grouped['ANOS_MONITORADOS'].min(),
                 df_grouped['ANOS_MONITORADOS'].max() + 1).tolist()

# Layout
fig.update_layout(
    height=600,
    barmode="stack",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-.15,
        xanchor="left",
        x=0,
        title="UF: "
    ),
    xaxis=dict(
        type='linear',
        title="Ano",
        tickmode='array',
        tickvals=anos,       # força mostrar todos os anos
        ticktext=anos
    ),
    xaxis_range=[df_grouped['ANOS_MONITORADOS'].min() - 0.5, 2024.5],
    
    margin=dict(r=120),
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        x=1.15,
        y=1.05,
        showactive=True,
        
    )],
    title=f"Monitoramento de {poluentes[0]}"
)

HTML(fig.to_html(include_plotlyjs="cdn"))


[1992.5, 2024.5]